In [26]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Section A - Coding

In this section, we implement Decision Tree and Random Forest models from scratch for both classification and regression.

**Node Class** - this class represents a single node in the tree.   It stores the node impurity, the prediction, the split rule, and links to the child nodes.  
The same class is used for both classification and regression trees.

In [27]:
# Class for one node in a tree
class Node:
    def __init__(
        self,
        impurity=None,
        num_samples=None,
        prediction=None,
        num_samples_per_class=None,
        feature_index=None,
        threshold=None,
        left=None,
        right=None
    ):
        # impurity at this node
        self.impurity = impurity

        # number of samples in this node
        self.num_samples = num_samples

        # stored prediction:
        self.prediction = prediction

        # class counts (used in classification if needed)
        self.num_samples_per_class = num_samples_per_class

        # split rule
        self.feature_index = feature_index
        self.threshold = threshold

        # children
        self.left = left
        self.right = right

    def is_leaf_node(self):
        return self.left is None and self.right is None

**Gini Impurity** - this function computes the impurity of a node in the classification tree.  
It measures how mixed the class labels are inside the node.  
Formula: **Gini = 1 - Σ(pᵢ²)**.

In [28]:
# Gini impurity calculation function
def calculate_gini(y):
    n = len(y)

    if n == 0:
        return 0.0

    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / n

    return 1.0 - np.sum(probabilities ** 2)

**SSR Calculation** - this function computes the impurity of a node in the regression tree.  
It measures how far the target values are from their mean inside the node.  
Formula: **SSR = Σ(yᵢ - ȳ)²**.

In [29]:
# SSR calculation
def calculate_ssr(y):
    n = len(y)

    if n == 0:
        return 0.0

    mean_value = np.mean(y)
    ssr = np.sum((y - mean_value) ** 2)

    return ssr

**Best Split** - this function searches for the best feature and threshold to split the node. For classification it uses Gini gain, and for regression it uses SSR reduction.  
It also enforces the minimum samples per leaf rule.

In [30]:
# Find the best split for classification or regression
def find_best_split(X, y, feature_names, min_samples_leaf=5, is_classifier=True):
    n_samples, n_features = X.shape

    if n_samples <= 1:
        return None, None, 0.0

    # parent impurity
    if is_classifier:
        parent_score = calculate_gini(y)
    else:
        parent_score = calculate_ssr(y)

    best_gain = 0.0
    best_feature_index = None
    best_threshold = None

    # sort features alphabetically
    sorted_feature_indices = sorted(range(n_features), key=lambda i: feature_names[i])

    for feature_index in sorted_feature_indices:
        values = X[:, feature_index]
        unique_values = np.unique(values)

        # no split possible
        if len(unique_values) < 2:
            continue

        # candidate thresholds
        thresholds = (unique_values[:-1] + unique_values[1:]) / 2

        for threshold in thresholds:
            left_mask = values <= threshold
            right_mask = values > threshold

            # skip small leaves
            if left_mask.sum() < min_samples_leaf or right_mask.sum() < min_samples_leaf:
                continue

            left_y = y[left_mask]
            right_y = y[right_mask]

            if is_classifier:
                left_score = calculate_gini(left_y)
                right_score = calculate_gini(right_y)

                child_score = (
                    (len(left_y) / n_samples) * left_score
                    + (len(right_y) / n_samples) * right_score
                )
            else:
                left_score = calculate_ssr(left_y)
                right_score = calculate_ssr(right_y)

                child_score = left_score + right_score

            gain = parent_score - child_score

            # keep the first alphabetical feature
            if gain > best_gain:
                best_gain = gain
                best_feature_index = feature_index
                best_threshold = threshold

    return best_feature_index, best_threshold, best_gain

In [2]:
# Helper functions for Random Forest

import numpy as np


def bootstrap_sample(X, y):
    """
    Create a bootstrap sample (sampling with replacement).
    The sample has the same size as the original dataset.
    Works with pandas DataFrame and Series.
    """
    n_samples = X.shape[0]

    # sample row indices with replacement
    indices = np.random.choice(n_samples, size=n_samples, replace=True)

    X_resampled = X.iloc[indices]
    y_resampled = y.iloc[indices]

    return X_resampled, y_resampled

def majority_vote(predictions):
    """
    Return the most common class among predictions.
    Used in classification.
    """
    values, counts = np.unique(predictions, return_counts=True)
    return values[np.argmax(counts)]


def average_predictions(predictions):
    """
    Return the average of predictions.
    Used in regression.
    """
    return np.mean(predictions)